In [8]:
import pandas as pd
import numpy as np

In [9]:
X_train_scaled = pd.read_csv(r"data\cleaned\X_train_scaled.csv", index_col=0)
X_val_scaled = pd.read_csv(r"data\cleaned\X_val_scaled.csv", index_col=0)
y_train = pd.read_csv(r"data\cleaned\y_train.csv", index_col=0)["Target"]
y_val = pd.read_csv(r"data\cleaned\y_val.csv", index_col=0)["Target"]

In [10]:
X_train_scaled

,Marital status,Application mode,Application order,Course,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,...,Scholarship holder,Age at enrollment,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade)
2930,-0.288731,-0.076465,-0.558954,0.391673,-0.351240,-0.431623,-1.051856,-1.254494,-0.372292,-0.388466,...,-0.589618,-0.678920,-0.107284,-2.015935,-1.537585,-2.209834,-0.099064,-2.053900,-1.465463,-1.960188
527,-0.288731,-0.076465,-0.558954,0.441428,-0.351240,3.320607,-1.116316,-1.385174,4.243212,4.823940,...,1.696013,-0.678920,-0.107284,-0.557906,0.422846,1.030973,-0.099064,-0.511815,0.521190,0.691171
1342,-0.288731,1.187659,-0.558954,-0.408271,-0.351240,0.026149,-1.180777,-0.209049,-0.220964,-0.271771,...,1.696013,0.503715,-0.107284,1.872141,-1.210847,-0.141234,-0.099064,1.287285,-1.465463,-1.960188
2046,-0.288731,1.474960,-0.558954,0.130821,3.349296,-0.206489,-1.180777,-1.254494,-0.069636,-0.349568,...,1.696013,-0.284708,-0.107284,1.143127,-0.557369,0.341439,-0.099064,1.030271,-1.134354,0.531450
2890,-0.288731,-0.076465,-0.558954,0.125508,-0.351240,-0.131444,-1.180777,-1.254494,-0.069636,-0.038379,...,-0.589618,-0.416112,-0.512345,-0.314901,-0.230631,0.272486,-0.551873,0.002214,-0.803245,0.435618
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,-0.288731,1.474960,-0.558954,0.190721,3.349296,2.795295,-0.020487,-1.319834,-0.069636,-0.310669,...,-0.589618,0.372311,0.702838,1.386132,1.076324,0.918923,-0.099064,1.030271,0.521190,0.659227
192,-0.288731,-0.995828,-0.558954,0.069473,-0.351240,0.026149,1.204262,1.032416,-0.258796,-0.116176,...,-0.589618,-0.284708,2.323083,1.143127,2.383278,0.797592,2.617787,1.544299,2.507842,0.811575
48,-0.288731,-0.995828,-0.558954,0.441428,-0.351240,-1.182069,-1.180777,0.967075,-0.220964,-0.077278,...,1.696013,-0.547516,-0.107284,-0.071897,0.422846,0.238009,-0.099064,0.773256,-0.472136,0.531450
2667,-0.288731,1.417500,-0.558954,-0.408271,0.719968,-1.407203,0.946420,0.771055,-0.410124,3.423592,...,-0.589618,0.766522,-0.107284,0.414113,0.422846,0.203533,-0.099064,-0.254800,0.521190,0.148121


In [11]:
y_train

2930    0
527     2
1342    0
2046    0
2890    1
       ..
154     1
192     2
48      2
2667    2
1500    0
Name: Target, Length: 2322, dtype: int64

# Decision Tree

Constructed using OOP paradigm to mimics the sklearn usage & support multiple instatiation for random forest and future ensemble

> The implementation of ID3 DTL just faithfully follows and implement this tutorial: https://medium.com/@MalikSaadAhmed/decision-tree-classifier-using-numpy-and-skit-learn-514028f71658

The tutorial already adopt few of the other DTL concepts:
* Forcefully split feature values into left-right (binary) instead of considering each unique values (CART). I decided to not modify the binary split code because the dataset used for this project is already encoded to numeric, and there will be lots of children of that subtree. Thus I agree with the tutorial to borrow CART binary split concept. 

While some of the ID3 concepts remains:
* Considering IG for the root,

According to Hssina, B., et al. (2014). A comparative study of decision tree ID3 and C4.5. International Journal of Advanced Computer Science and Applications, 4(2). https://doi.org/10.14569/SpecialIssue.2014.040203, C4.5 can be achieved by extending the properties:
* Calculate split_info $Split\ Information = -\Sigma_{c = 1}^{C}p_clog_2(p_c)$
* Consider $Gain\ Ratio = \frac{Information\ Gain}{Split\ Information}$ to be the root / subtree. 

Other sauce:
* https://scientistcafe.com/ids/splitting-criteria#information-gain-ratio-igr

In [12]:
class DecisionTree:
    def __init__(self, max_depth = None, min_samples_split = 0, max_features  = None):
        self.tree = None # weights
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.feature_names = None

    # reusable: calculate entropy for (one) value of a feature, according to the target proportion
    def entropy(self, y):
        no_instances = len(y)
        unique_values = y.unique()
        entropy = 0
        for value in unique_values:
            no_values = len(y[y == value])
            proportion = no_values / no_instances
            entropy -= proportion * np.log2(proportion)
        return entropy

    # reusable: calculate IG of a splitted feature
    def information_gain(self, data, feature, y, threshold):
        entropy_S = self.entropy(data[y])
        left = data[data[feature]<=threshold]
        right = data[data[feature]>threshold]
        weighted_entropy = (len(left)/len(data))*self.entropy(left[y]) + (len(right)/len(data))*self.entropy(right[y])
        return  entropy_S - weighted_entropy

    def calc_split_info(self, data, feature, threshold):
        p_left = (data[feature] <= threshold).mean()
        split_info = 0
        for p in (p_left, 1-p_left):
            if p > 0:
                split_info -= p * np.log2(p)
        return split_info
        

    # finding optimal slicing point for a feature
    def numeric_thresholding(self, df, feature, y):
        values = np.sort(df[feature].unique())
        # unique value handler
        if len(values) < 2:
            return None, 0
        slicing_candidates = (values[:-1]+values[1:])/2 # a list of possible values to be a slicing [L/R] threshold
        best_th, best_gain = None, -1
        # iterate through each slicing candidate as the threshold
        for t in slicing_candidates:
            gain = self.information_gain(df, feature, y,t)
            if gain > best_gain:
                best_th, best_gain = t, gain
        return best_th, best_gain

    # chosing best features based on IG (highest)
    # target for modification ID3 -> C4.5: choose_best_feature criteria
    # shift from plain IG to Gain Ratio
    def choose_best_feature(self, data, features, y):

        # for RF --> random feature subsampling mechanism
        # making each recursive calls from build method narrows available passed feature into randomly chosen feature of size k
        if self.max_features is not None:
            k = min(self.max_features, len(features))
            features = np.random.choice(features, size=k, replace=False)

        features_thresh_gains = {}
        for feature in features:
            th,gain = self.numeric_thresholding(data, feature, y)
            if th is not None:
                features_thresh_gains[feature] = (th, gain)
        gain_ratios = {}
        for feature, (th, gain) in features_thresh_gains.items():
            split_info = self.calc_split_info(data, feature, th)
            if split_info > 0:
                gain_ratios[feature] = gain / split_info
        if not gain_ratios:
            return None, None
        
        best_feature = max(gain_ratios, key=gain_ratios.get)
        return best_feature, features_thresh_gains[best_feature][0]
        


    # force split into left right
    def split(self, df, feature, threshold):
        left = df[df[feature]<=threshold]
        right = df[df[feature]>threshold]
        return left,right

    # recursive function, 
    # smaller problem: creating decision stump over and over
    # base: maxdepth reached, (modify, target): minimum data not reached
    # recursive: evaluating IG of all features and split   
    def build(self,df, features, target, depth = 0):
        if self.max_depth is not None and depth >= self.max_depth:#maxdepth reached
            return df[target].mode().iloc[0] # majority vote
        if len(np.unique(df[target]))==1: # pure node
            return df[target].iloc[0]
        if len(df)<self.min_samples_split:#minimum split escape
            return df[target].mode().iloc[0] # majority vote

        node = {}
        best_feature, threshold = self.choose_best_feature(df, features, target)
        if best_feature is None:
            return df[target].mode().iloc[0] # majority vote

        node['feature'] = best_feature
        node['children'] = {}
        node['split_value'] = threshold
        left, right = self.split(df, best_feature, threshold)
        node['children']['left'] = self.build(left, features, target, depth + 1)
        node['children']['right'] = self.build(right, features, target, depth + 1)
        return node
            
    def fit(self, X, y):
        # per-row evaluation, previously detached X,y data is merged again
        self.feature_names = X.columns
        merged_df = X.copy()
        merged_df['target'] = y
        features = merged_df.columns[:-1]
        target = merged_df.columns[-1]
        self.tree = self.build(merged_df, features, target)

    def predict_instance(self, X, tree):
        # loop until the children is a label
        while isinstance(tree, dict): 
            feature = tree['feature']
            # match condition of the feature value of an instance
            if X[feature] <= tree['split_value']:
                tree = tree['children']['left']
            else:
                tree = tree['children']['right']
        return tree

    # predict data
    def predict(self,X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns = self.feature_names)

        predictions = []
        for _, row in X.iterrows():
            predictions.append(self.predict_instance(row, self.tree))
        return predictions


In [13]:
dtl1 = DecisionTree(max_depth = 1)
dtl1.fit(X_train_scaled, y_train)

In [14]:
y_val_pred = dtl1.predict(X_val_scaled)

In [15]:
from sklearn.metrics import accuracy_score
acc_val_dtl = accuracy_score(y_val,y_val_pred)
print(acc_val_dtl)

0.7041343669250646


### Save Model

In [16]:
import joblib 

joblib.dump(dtl1, '../output/dtl1.pkl')

['../output/dtl1.pkl']

In [17]:
loaded_dtl = joblib.load('../output/dtl1.pkl')
print(loaded_dtl)
y_val_pred_loaded_model = loaded_dtl.predict(X_val_scaled)
acc_y_val_pred_loaded_model = accuracy_score(y_val,y_val_pred_loaded_model)
print(acc_y_val_pred_loaded_model)

0.7041343669250646


# Random Forest

Elementary one, consists of these properties:
* Bootstrap sampling of the data
* Random feature sampling (embedded on DTL code)

In [18]:
class RandomForest:
    def __init__(self, no_tree = 10, max_depth = 3, min_samples_split = 0):
        self.forest = []
        self.no_tree = no_tree
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_names = None

    def fit(self,X,y):
        # need: bootstraping data, feature sampling, flexible according to the number of tree
        # bootstraping data: 
        self.feature_names = X.columns
        merged_df = X.copy()
        merged_df['target'] = y
        feature_subsampling_sz = max(1,int(np.sqrt(X.shape[1])))
        
        for i in range(self.no_tree):
            print(f"\rTree {i + 1} of {self.no_tree}...", end="", flush=True)
            # bootstrap sampling using pandas
            # frac 1 results 100% df length, 
            bagging_df = merged_df.sample(frac=1.0, replace=True).reset_index(drop=True)
            bagging_df_X = bagging_df.drop(columns=['target'])
            bagging_df_y = bagging_df['target']

            tree = DecisionTree(max_depth = self.max_depth, min_samples_split=self.min_samples_split, max_features = feature_subsampling_sz)
            tree.fit(bagging_df_X, bagging_df_y)
            self.forest.append(tree)

    def predict_instance(self,X,forest):
        vote = []
        for tree in forest:
            vote.append(tree.predict_instance(X,tree.tree))
        # majority voting
        return pd.Series(vote).mode().iloc[0] # majority vote

        
    def predict(self,X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names)
        predictions = []
        for _, row in X.iterrows():
            predictions.append(self.predict_instance(row, self.forest))
        return predictions

In [19]:
rf1 = RandomForest()
rf1.fit(X_train_scaled, y_train)


Tree 1 of 10...

Tree 10 of 10...

In [20]:
rf1_y_val_pred = rf1.predict(X_val_scaled)

In [21]:
from sklearn.metrics import accuracy_score
acc_val_rf = accuracy_score(y_val,rf1_y_val_pred)
print(acc_val_rf)

0.7157622739018088


In [23]:
import joblib 

joblib.dump(rf1, '../output/rf1.pkl')

['../output/rf1.pkl']

# Predicting Test Instances

In [25]:
test_df = pd.read_csv('data/cleaned/test_df.csv', index_col=0)
student_ids = test_df['Student_ID'].to_numpy()

test_X = test_df.loc[:, X_train_scaled.columns]
target_mapping = {0: 'Dropout', 1: 'Enrolled', 2: 'Graduate'}


def decode_targets(predictions):
    labels = pd.Series(predictions).replace(target_mapping)
    if not labels.isin(target_mapping.values()).all():
        raise ValueError('Unknown target label in predictions.')
    return labels.to_numpy()


rf1_test_predict = rf1.predict(test_X.values)
rf1_submit = pd.DataFrame({'Student_ID': student_ids, 'Target': decode_targets(rf1_test_predict)})
rf1_submit.to_csv(f'../output/csv/submission_random_forest.csv', index=False)